In [1]:
import numpy as np
from scipy.stats import norm

RUNS       = 5000
ALPHA      = 0.05
P          = 50
K          = 3
N_I        = 25
SEED       = 20250831
EPS        = 1e-12

def _Bi_within(X):
    n = X.shape[0]
    if n < 2:
        return 0.0
    G = X @ X.T
    np.fill_diagonal(G, 0.0)
    s = np.sum(G * G)
    return s / (n * (n - 1))

def _Bij_cross(X, Y):
    H = X @ Y.T
    s = np.sum(H * H)
    return s / (X.shape[0] * Y.shape[0])

def ahmad_Tg_test(data_groups, alpha=ALPHA):
    g = len(data_groups)
    n_i = [grp.shape[0] for grp in data_groups]
    p = data_groups[0].shape[1]

    B_within = [ _Bi_within(grp) for grp in data_groups ]
    B_pairs = {}
    for i in range(g):
        for j in range(i+1, g):
            Bij = _Bij_cross(data_groups[i], data_groups[j])
            B_pairs[(i, j)] = Bij

    sum_Bi  = float(np.sum(B_within))
    sum_Bij = float(np.sum(list(B_pairs.values())))
    Tg = ((g - 1) * sum_Bi - 2.0 * sum_Bij) / (p * p)

    P_star = 0.0
    numerator_C3 = 0.0
    for i in range(g):
        for j in range(g):
            if i == j:
                continue
            ni, nj = n_i[i], n_i[j]
            key = (i, j) if i < j else (j, i)
            Bij = B_pairs[key]
            numerator_C3 += ni * nj * Bij
            P_star += ni * nj
    C3 = numerator_C3 / max(P_star, EPS)

    term1 = (g - 1) ** 2 * np.sum([1.0 / (ni ** 2) for ni in n_i])
    term2 = 0.0
    for i in range(g):
        for j in range(i+1, g):
            term2 += 2.0 / (n_i[i] * n_i[j])
    A = term1 + term2

    sigma_hat = 2.0 * np.sqrt(max(A, 0.0)) * (C3 / (p * p))
    sigma_hat = max(sigma_hat, EPS)

    Z = Tg / sigma_hat
    crit = norm.ppf(1 - alpha)
    return Z, crit, 0

#AR(1)
def ar1_matrix(p, rho):
    idx = np.arange(p)
    return rho ** np.abs(np.subtract.outer(idx, idx))

def simulate_groups_AR1_H0(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    F0 = ar1_matrix(p, rho=0.7)
    means = [np.zeros(p) for _ in range(k)]
    return [rng.multivariate_normal(means[i], F0, n_i) for i in range(k)]

def simulate_groups_AR1_H1_diff(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    F0 = ar1_matrix(p, rho=0.7)
    F1 = ar1_matrix(p, rho=0.9)
    means = [np.zeros(p) for _ in range(k)]
    groups = []
    for i in range(k):
        Sigma_i = F1 if i == (k - 1) else F0
        groups.append(rng.multivariate_normal(means[i], Sigma_i, n_i))
    return groups, (k - 1)

#CS
def compound_symmetry_matrices(p):
    I = np.eye(p); J = np.ones((p, p))
    return 0.99 * I + 0.01 * J, 0.90 * I + 0.10 * J

def simulate_groups_CS_H0(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    C0, _ = compound_symmetry_matrices(p)
    means = [np.zeros(p) for _ in range(k)]
    return [rng.multivariate_normal(means[i], C0, n_i) for i in range(k)]

def simulate_groups_CS_H1_diff(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    C0, C1 = compound_symmetry_matrices(p)
    means = [np.zeros(p) for _ in range(k)]
    groups = []
    for i in range(k):
        Sigma_i = C1 if i == (k - 1) else C0
        groups.append(rng.multivariate_normal(means[i], Sigma_i, n_i))
    return groups, (k - 1)

#SIM
def simulate_groups_SIM_H0(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    S0 = np.eye(p)
    means = [np.zeros(p) for _ in range(k)]
    return [rng.multivariate_normal(means[i], S0, n_i) for i in range(k)]

def simulate_groups_SIM_H1_diff(p=P, k=K, n_i=N_I, rng=None, S1=None):
    if rng is None:
        rng = np.random.default_rng()
    S0 = np.eye(p)
    if S1 is None:
        u = rng.uniform(2.0, 3.0, size=p)
        S1 = np.diag(u)
    means = [np.zeros(p) for _ in range(k)]
    groups = []
    for i in range(k):
        Sigma_i = S1 if i == (k - 1) else S0
        groups.append(rng.multivariate_normal(means[i], Sigma_i, n_i))
    return groups, (k - 1)

#BTOEP
def toeplitz_matrix(p):
    T0 = np.zeros((p, p)); T1 = np.zeros((p, p))
    np.fill_diagonal(T0, 1.0)
    i = np.arange(p - 1)
    T0[i, i + 1] = T0[i + 1, i] = -0.5
    np.fill_diagonal(T1, 1.0)
    T1[i, i + 1] = T1[i + 1, i] = -0.05
    return T0, T1

def simulate_groups_TOEP_H0(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    T0, _ = toeplitz_matrix(p)
    means = [np.zeros(p) for _ in range(k)]
    return [rng.multivariate_normal(means[i], T0, n_i) for i in range(k)]

def simulate_groups_TOEP_H1_diff(p=P, k=K, n_i=N_I, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    T0, T1 = toeplitz_matrix(p)
    means = [np.zeros(p) for _ in range(k)]
    groups = []
    for i in range(k):
        Sigma_i = T1 if i == (k - 1) else T0
        groups.append(rng.multivariate_normal(means[i], Sigma_i, n_i))
    return groups, (k - 1)

#VC
def variance_components(p, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    return np.diag(rng.uniform(1, 2, size=p)), np.diag(rng.uniform(3, 4, size=p))

def simulate_groups_VC_H0(p=P, k=K, n_i=N_I, rng=None, A0=None):
    if rng is None:
        rng = np.random.default_rng()
    if A0 is None:
        A0, _ = variance_components(p, rng=rng)
    means = [np.zeros(p) for _ in range(k)]
    return [rng.multivariate_normal(means[i], A0, n_i) for i in range(k)]

def simulate_groups_VC_H1_diff(p=P, k=K, n_i=N_I, rng=None, A0=None, A1=None):
    if rng is None:
        rng = np.random.default_rng()
    if (A0 is None) or (A1 is None):
        A0_new, A1_new = variance_components(p, rng=rng)
        A0 = A0 if A0 is not None else A0_new
        A1 = A1 if A1 is not None else A1_new
    means = [np.zeros(p) for _ in range(k)]
    groups = []
    for i in range(k):
        Sigma_i = A1 if i == (k - 1) else A0
        groups.append(rng.multivariate_normal(means[i], Sigma_i, n_i))
    return groups, (k - 1)

#Monte Carlo
def monte_carlo(runs=RUNS, alpha=ALPHA, p=P, k=K, n_i=N_I, seed=SEED, structure="AR1"):
    rng = np.random.default_rng(seed)
    S1_fix = None
    A0_fix = None
    A1_fix = None
    if structure == "SIM":
        u_fix  = rng.uniform(2.0, 3.0, size=p)
        S1_fix = np.diag(u_fix)
    elif structure == "VC":
        A0_fix, A1_fix = variance_components(p, rng=rng)

    reject_H0_false = 0
    reject_H0_true  = 0

    for _ in range(runs):
        if structure == "AR1":
            groups0 = simulate_groups_AR1_H0(p=p, k=k, n_i=n_i, rng=rng)
            stat0, crit0, _ = ahmad_Tg_test(groups0, alpha=alpha)
            if stat0 > crit0: reject_H0_false += 1

            groups1, _ = simulate_groups_AR1_H1_diff(p=p, k=k, n_i=n_i, rng=rng)
            stat1, crit1, _ = ahmad_Tg_test(groups1, alpha=alpha)
            if stat1 > crit1: reject_H0_true += 1

        elif structure == "CS":
            groups0 = simulate_groups_CS_H0(p=p, k=k, n_i=n_i, rng=rng)
            stat0, crit0, _ = ahmad_Tg_test(groups0, alpha=alpha)
            if stat0 > crit0: reject_H0_false += 1

            groups1, _ = simulate_groups_CS_H1_diff(p=p, k=k, n_i=n_i, rng=rng)
            stat1, crit1, _ = ahmad_Tg_test(groups1, alpha=alpha)
            if stat1 > crit1: reject_H0_true += 1

        elif structure == "SIM":
            groups0 = simulate_groups_SIM_H0(p=p, k=k, n_i=n_i, rng=rng)
            stat0, crit0, _ = ahmad_Tg_test(groups0, alpha=alpha)
            if stat0 > crit0: reject_H0_false += 1

            groups1, _ = simulate_groups_SIM_H1_diff(p=p, k=k, n_i=n_i, rng=rng, S1=S1_fix)
            stat1, crit1, _ = ahmad_Tg_test(groups1, alpha=alpha)
            if stat1 > crit1: reject_H0_true += 1

        elif structure == "TOEP":
            groups0 = simulate_groups_TOEP_H0(p=p, k=k, n_i=n_i, rng=rng)
            stat0, crit0, _ = ahmad_Tg_test(groups0, alpha=alpha)
            if stat0 > crit0: reject_H0_false += 1

            groups1, _ = simulate_groups_TOEP_H1_diff(p=p, k=k, n_i=n_i, rng=rng)
            stat1, crit1, _ = ahmad_Tg_test(groups1, alpha=alpha)
            if stat1 > crit1: reject_H0_true += 1

        elif structure == "VC":
            groups0 = simulate_groups_VC_H0(p=p, k=k, n_i=n_i, rng=rng, A0=A0_fix)
            stat0, crit0, _ = ahmad_Tg_test(groups0, alpha=alpha)
            if stat0 > crit0: reject_H0_false += 1

            groups1, _ = simulate_groups_VC_H1_diff(p=p, k=k, n_i=n_i, rng=rng, A0=A0_fix, A1=A1_fix)
            stat1, crit1, _ = ahmad_Tg_test(groups1, alpha=alpha)
            if stat1 > crit1: reject_H0_true += 1

    return {
        "structure": structure,
        "alpha": alpha,
        "runs": runs,
        "df": 0,
        "empirical_size": reject_H0_false / runs,
        "empirical_power": reject_H0_true / runs,
        "beta_global": 1 - (reject_H0_true / runs),
    }

if __name__ == "__main__":
    for struct in ["AR1", "CS", "SIM", "TOEP", "VC"]:
        res = monte_carlo(structure=struct)
        print(f"==== Monte Carlo ({res['structure']} structure) ====")
        print(f"alpha = {res['alpha']} | runs = {res['runs']}")
        print(f"Empirical size : {res['empirical_size']:.4f}")
        print(f"Empirical power: {res['empirical_power']:.4f}")
        print(f"Beta : {res['beta_global']:.4f}\n")

==== Monte Carlo (AR1 structure) ====
alpha = 0.05 | runs = 5000
Empirical size : 0.0736
Empirical power: 0.8476
Beta : 0.1524

==== Monte Carlo (CS structure) ====
alpha = 0.05 | runs = 5000
Empirical size : 0.0612
Empirical power: 0.5700
Beta : 0.4300

==== Monte Carlo (SIM structure) ====
alpha = 0.05 | runs = 5000
Empirical size : 0.0614
Empirical power: 0.9992
Beta : 0.0008

==== Monte Carlo (TOEP structure) ====
alpha = 0.05 | runs = 5000
Empirical size : 0.0626
Empirical power: 0.5898
Beta : 0.4102

==== Monte Carlo (VC structure) ====
alpha = 0.05 | runs = 5000
Empirical size : 0.0660
Empirical power: 0.9908
Beta : 0.0092

